# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#Library setting that makes our inspection easy
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

In [2]:
#DataPath of the folder that contain all the tables/files of dataset
DATA_PATH = Path("D:\Profit_leak_growth_Analysis\Dataset & Readme")

In [3]:
#Load all the tables
customers = pd.read_csv(DATA_PATH / "raw_customers.csv")
products = pd.read_csv(DATA_PATH / "raw_products.csv")
stores = pd.read_csv(DATA_PATH / "raw_stores.csv")
campaigns = pd.read_csv(DATA_PATH / "raw_campaigns.csv")
orders = pd.read_csv(DATA_PATH / "raw_orders.csv")
order_items = pd.read_csv(DATA_PATH / "raw_order_items.csv")
returns = pd.read_csv(DATA_PATH / "raw_returns.csv")
shipments = pd.read_csv(DATA_PATH / "raw_shipments.csv")
marketing = pd.read_csv(DATA_PATH / "raw_marketing_daily.csv")
date_dim = pd.read_csv(DATA_PATH / "dim_date.csv")

In [4]:
#Check the tables that they load successfully
products.head()

,product_id,product_name,category,subcategory,brand,supplier_id,list_price,standard_cost,launch_date,product_status
0,P00001,Crest Yoga 0001,Sports,Yoga,Crest,SUP056,122.12,74.34,2023-06-18,Active
1,P00002,Everline Skincare 0002,Beauty,Skincare,Everline,SUP058,40.86,21.23,2023-06-16,Active
2,P00003,UrbanPeak Sneakers 0003,Footwear,Sneakers,UrbanPeak,SUP030,58.19,32.47,2023-07-25,Active
3,P00004,Aster T-Shirts 0004,Apparel,T-Shirts,Aster,SUP053,94.82,49.10,2023-11-03,Active
4,P00005,Everline Speakers 0005,Electronics,Speakers,Everline,SUP044,16.58,4.03,2023-07-13,Active


In [5]:
#Confirm that Python read everything correctly

#Create dictonary
tables = {
    "raw_customers": customers,
    "raw_products": products,
    "raw_stores": stores,
    "raw_campaigns": campaigns,
    "raw_orders": orders,
    "raw_order_items": order_items,
    "raw_returns": returns,
    "raw_shipments": shipments,
    "raw_marketing_daily": marketing,
    "dim_date": date_dim
}

#Check no of rows & column in each table
for name, df in tables.items():
    print(f"{name}: {df.shape}")

raw_customers: (30000, 10)
raw_products: (1000, 10)
raw_stores: (30, 8)
raw_campaigns: (60, 7)
raw_orders: (75000, 11)
raw_order_items: (105000, 10)
raw_returns: (14479, 11)
raw_shipments: (30000, 8)
raw_marketing_daily: (7965, 8)
dim_date: (1520, 10)


# Audit Table Structure

We are doing this to answer: "How Large is every table and does it contain exact duplicate rows?"

In [6]:
#Table Structure
structure_audit=[]

for name,df in tables.items():
    structure_audit.append({
        'table': name,
        'rows':len(df),
        'columns':len(df.columns),
        'duplicated_rows':df.duplicated().sum() }
                          )
structure_audit=pd.DataFrame(structure_audit)
structure_audit

,table,rows,columns,duplicated_rows
0,raw_customers,30000,10,0
1,raw_products,1000,10,0
2,raw_stores,30,8,0
3,raw_campaigns,60,7,0
4,raw_orders,75000,11,0
5,raw_order_items,105000,10,0
6,raw_returns,14479,11,0
7,raw_shipments,30000,8,0
8,raw_marketing_daily,7965,8,0
9,dim_date,1520,10,0


 As we can see,There is no duplicate value in any of our table.

# Audit: Data Types

In [7]:
for name, df in tables.items():
    print("\n" + "=" *60)
    print(name)
    print("=" * 60)
    print(df.dtypes)
    


raw_customers
customer_id            object
signup_date            object
acquisition_channel    object
customer_type          object
country                object
state                  object
city                   object
age_band               object
income_band            object
loyalty_tier           object
dtype: object

raw_products
product_id         object
product_name       object
category           object
subcategory        object
brand              object
supplier_id        object
list_price        float64
standard_cost     float64
launch_date        object
product_status     object
dtype: object

raw_stores
store_id           object
store_name         object
region             object
state              object
city               object
store_type         object
store_size_sqft     int64
opened_date        object
dtype: object

raw_campaigns
campaign_id       object
campaign_name     object
channel           object
campaign_type     object
objective         object
start_dat

We need to fix some datatypes such as "Orderdate" to datetime instead of object. Most of the issues are in datefixing, other are almost good. Only few datatype errors

# Audit:Missing Values

In [8]:
missing_audit= []

for name,df in tables.items():

    for column in df.columns:
        missing_count= df[column].isna().sum()
        missing_pct= missing_count / len(df) * 100

        missing_audit.append({
            'table':name,
            'column':column,
            'missing_count':missing_count,
            'missing_pct':round(missing_pct,2)
        })
        
missing_audit=pd.DataFrame(missing_audit)

missing_audit=missing_audit[
    missing_audit['missing_count'] > 0].sort_values(['missing_pct', 'missing_count'], ascending= False )

missing_audit

,table,column,missing_count,missing_pct
38,raw_orders,store_id,56358,75.14
43,raw_orders,coupon_code,37660,50.21
6,raw_customers,city,240,0.80
42,raw_orders,payment_method,450,0.60
8,raw_customers,income_band,90,0.30
78,raw_marketing_daily,clicks,7,0.09


# Audit:Primary Key uniqueness

In [9]:
#Primary key of each table:

primary_keys={
        "raw_customers":"customer_id",
        "raw_products":"product_id",
        "raw_stores":"store_id",
        "raw_campaigns":"campaign_id",
        "raw_orders":"order_id",
        "raw_order_items":"order_item_id",
        "raw_returns":"return_id",
        "raw_shipments":"shipment_id"
    }

#Audit
pk_audit = []

for table_name, pk in primary_keys.items():

    df = tables[table_name]

    duplicate_count = df[pk].duplicated().sum()
    null_count = df[pk].isna().sum()

    pk_audit.append({
        "table": table_name,
        "primary_key": pk,
        "duplicate_keys": duplicate_count,
        "null_keys": null_count,
        "status": "PASS"
        if duplicate_count == 0 and null_count == 0
        else "FAIL"
    })

pk_audit = pd.DataFrame(pk_audit)

pk_audit


,table,primary_key,duplicate_keys,null_keys,status
0,raw_customers,customer_id,0,0,PASS
1,raw_products,product_id,0,0,PASS
2,raw_stores,store_id,0,0,PASS
3,raw_campaigns,campaign_id,0,0,PASS
4,raw_orders,order_id,0,0,PASS
5,raw_order_items,order_item_id,0,0,PASS
6,raw_returns,return_id,0,0,PASS
7,raw_shipments,shipment_id,0,0,PASS


Hence prove there is no issue in primary key of any table.


# Audit:Foreign Key
Now we test whether our tables actually connect

In [63]:
orphan_orders = orders[~orders['customer_id'].isin(customers['customer_id'])]

print(f"Total number of orphan orders: {len(orphan_orders)}")


Total number of orphan orders: 0


No orphan_order, no issue

In [65]:
#Order Items → Orders
orphan_order_items= order_items[~order_items['order_id'].isin(orders['order_id'])]

len(orphan_order_items)

0

No orphan order_item

In [66]:
#Order Items → Products
orphan_products=order_items[
    ~order_items['product_id'].isin(products['product_id'])
                     ]

len(orphan_products)

0

No orphan_products

In [13]:
#Returns → Order Items
orphan_returns = returns[
    ~returns["order_item_id"].isin(order_items["order_item_id"])
]

len(orphan_returns)

0

In [14]:
#Shipments → Orders
orphan_shipments = shipments[
    ~shipments["order_id"].isin(orders["order_id"])
]

len(orphan_shipments)

0

In [15]:
#Marketing → Campaigns
orphan_marketing = marketing[
    ~marketing["campaign_id"].isin(campaigns["campaign_id"])
]

len(orphan_marketing)

0

### No Orphan value found.

# Audit: Date Integrity

In [16]:
#First convert dates temporarily for testing:
date_columns={
    'raw_customers':['signup_date'],
    'raw_products':['launch_date'],
    'raw_orders':['order_date'],
    'raw_returns':['return_date'],
    'raw_shipments':['ship_date','promised_date','delivery_date'],
    'raw_campaigns':['start_date'],
    'raw_marketing_daily':['date']
}

for table_name,columns in date_columns.items():

    df=tables[table_name]

    for column in columns:
        df[column]=pd.to_datetime(
            df[column],
            errors='coerce'
        )

# Test 1---Customer signup vs purchase

Business Rule:A customer should not place an order before registering

In [17]:
#Merge
order_customer_check=orders.merge(
    customers[['customer_id','signup_date']],
    on='customer_id',
    how='left'
)

#Check: It should be 0
invalid_signup_orders = order_customer_check[
    order_customer_check["order_date"] <
    order_customer_check["signup_date"]
]

len(invalid_signup_orders)

0

As we can see there is no invalid date in our data. Every order is done after registering not before that

# Test 2: Product launch date vs first sale

In [20]:
first_sale=(
    order_items.merge(
    orders[['order_id','order_date']],
    on='order_id',
    how='left' )
    .groupby('product_id')['order_date']
    .min()
    .reset_index(name='first_sale_date')

)

#Merge
product_launch_check = products.merge(
    first_sale,
    on="product_id",
    how="left"
)

#Invalid date
invalid_product_sales = product_launch_check[
    product_launch_check["first_sale_date"] <
    product_launch_check["launch_date"]
]

len(invalid_product_sales)

0

There is no issue in product launch date and sale date.

# Test 3---Shipment timing

In [21]:
shipment_order_check= shipments.merge(
    orders[['order_id','order_date']],
    on='order_id',
    how='left'
)

#Invalid date testing
invalid_ship_dates = shipment_order_check[
    shipment_order_check["ship_date"] <
    shipment_order_check["order_date"]
]

len(invalid_ship_dates)

0

# Test 4---Promised date

In [22]:
#Invallid date
invalid_promised_dates = shipments[
    shipments["promised_date"] <
    shipments["ship_date"]
]

len(invalid_promised_dates)

0

# Test 5:Delivery date

In [23]:
invalid_delivery_dates = shipments[
    shipments["delivery_date"] <
    shipments["ship_date"]
]

len(invalid_delivery_dates)

0

# Test 6 — Returns after purchase

In [24]:
#Merge
return_order_check = returns.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

#Invalid
invalid_returns = return_order_check[
    return_order_check["return_date"] <
    return_order_check["order_date"]
]

len(invalid_returns)



0

As we can see there is no issue in our dates. All the dates are clear.

# Audit #7: Quantity and financial logic

In [25]:
#Quantity cannot be negative
invalid_quantity=order_items[
    order_items['quantity'] <=0
]

len(invalid_quantity)

0

## Unit selling price vs list selling price

In [26]:
#unit selling price cannot be more than list price
invalid_prices = order_items[
    order_items["unit_selling_price"] >
    order_items["unit_list_price"]
]

len(invalid_prices)

0

## Discount reconciliation

It should be zero. Means no error

In [27]:
#Calculate
calculated_discount = (
    order_items["unit_list_price"] -
    order_items["unit_selling_price"]
)

# Compare
discount_difference = (
    order_items["discount_amount"] -
    calculated_discount
).abs()

#Check if there is any error in discount
(discount_difference > 0.01).sum()

np.int64(0)

There is no issue in discounts.

# Revenue reconciliation

Gross Revenue = Quantity × Unit Selling Price

In [28]:
#Gross Revenue
calculated_revenue = (
    order_items["quantity"] *
    order_items["unit_selling_price"]
)

#Compare
revenue_difference = (
    order_items["gross_revenue"] -
    calculated_revenue
).abs()

#Test if their is error or not
(revenue_difference > 0.01).sum()

np.int64(0)

# Product-cost reconciliation

Product Cost = Quantity × Unit Cost

In [31]:
#Calculate
calculated_cost = (
    order_items["quantity"] *
    order_items["unit_cost"]
)

#Compare
cost_difference = (
    order_items["product_cost"] -
    calculated_cost
).abs()

#Test
(cost_difference > 0.01).sum()

np.int64(0)

There is no issue

# Audit:Return quantity validation

We need to determine whether customers returned more units than they bought.

In [36]:
# Aggregate returns
returned_qty = (
    returns
    .groupby("order_item_id")["return_quantity"]
    .sum()
    .reset_index()
)

#Join it
return_quantity_check = order_items[
    ["order_item_id", "quantity"]
].merge(
    returned_qty,
    on="order_item_id",
    how="left"
)

#Missing return quantity means no return:
return_quantity_check["return_quantity"] = (
    return_quantity_check["return_quantity"]
    .fillna(0)
)

#Check the invalid returns
invalid_return_qty = return_quantity_check[
    return_quantity_check["return_quantity"] >
    return_quantity_check["quantity"]
]

len(invalid_return_qty)

51

51 order items violate the cumulative return-quantity rule. 51 order items have cumulative returned quantity greater than purchased quantity

# Audit: Order status vs returns

Returns should not be associated with cancelled orders.

In [38]:
# Merge
return_status_check = returns.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

#Test
invalid_return_status = return_status_check[
    return_status_check["order_status"] != "Completed"
]

len(invalid_return_status)

0

# Audit:Shipment status validation

We have:

* delivery_date
* promised_date
* delivery_status

We can independently calculate the expected status:

In [40]:
shipments["calculated_delivery_status"] = np.where(
    shipments["delivery_date"] <= shipments["promised_date"],
    "On Time",
    "Late"
)

#Compare
status_mismatch = shipments[
    shipments["delivery_status"] !=
    shipments["calculated_delivery_status"]
]

len(status_mismatch)

0

No issue in shipping status it is either "On-Time", or "Late'. Not any other option

# Audit:Marketing funnel checks

Now test:

Impressions
     &
Clicks

Clicks shouldn't exceed impressions.

In [41]:
invalid_clicks = marketing[
    marketing["clicks"] >
    marketing["impressions"]
]

len(invalid_clicks)

0

In [42]:
#Descprtive statistics of marketing funnel to check outliers
marketing[
    ["impressions", "clicks", "ad_spend",
     "new_customers", "attributed_orders",
     "attributed_revenue"]
].describe()

,impressions,clicks,ad_spend,new_customers,attributed_orders,attributed_revenue
count,7965.000000,7958.000000,7965.000000,7965.000000,7965.000000,7965.000000
mean,214572.546014,4425.655818,3265.486655,119.133961,222.932957,20185.106318
std,117548.004131,3505.409026,2600.041825,146.948922,266.286386,25234.602264
min,12052.000000,89.000000,75.640000,0.000000,1.000000,80.640000
25%,112762.000000,1845.750000,1405.890000,27.000000,52.000000,4520.570000
50%,214520.000000,3552.500000,2690.020000,68.000000,130.000000,11333.950000
75%,316192.000000,6059.000000,4323.880000,145.000000,274.000000,24768.200000
max,419992.000000,29668.000000,19981.760000,1316.000000,2412.000000,243959.270000


# Audit:Channel/store logic

This is another very useful business rule.

"Physical store orders should have a store."

In [43]:
physical_store_missing = orders[
    (orders["sales_channel"] == "Physical Store") &
    (orders["store_id"].isna())
]

len(physical_store_missing)

0

There is no physical store missing

In [44]:
digital_with_store = orders[
    (orders["sales_channel"] != "Physical Store") &
    (orders["store_id"].notna())
]

len(digital_with_store)

0

No issue here also.

In [48]:
physical_wrong_device = orders[
    (orders["sales_channel"] == "Physical Store") &
    (orders["device_type"] != "In-Store")
]

len(physical_wrong_device)

0

# Audit:Categorical consistency

In [50]:
orders["sales_channel"].value_counts(dropna=False)

sales_channel
Website           32156
Physical Store    18642
Mobile App        16653
Marketplace        7549
Name: count, dtype: int64

In [51]:
orders["shipping_method"].value_counts(dropna=False)

shipping_method
Standard    35935
Express     15704
Pickup      15296
Next-Day     8065
Name: count, dtype: int64

In [52]:
products["category"].value_counts(dropna=False)

category
Apparel           213
Home & Kitchen    181
Electronics       169
Accessories       130
Beauty            123
Footwear           95
Sports             89
Name: count, dtype: int64

In [53]:
customers["acquisition_channel"].value_counts(dropna=False)

acquisition_channel
Google Ads    6629
Meta          5949
Organic       5330
Direct        4801
Email         4174
Affiliate     3004
Google_Ads      12
GOOGLE ADS      11
ORGANIC         10
DIRECT          10
meta            10
google ads      10
email            9
direct           9
META             9
EMAIL            8
organic          8
affiliate        4
AFFILIATE        3
Name: count, dtype: int64

Standardization required in acquisition_channel during cleaning.